# LeetCode 593 — Valid Square bằng Python

Link đề: https://leetcode.com/problems/valid-square/ · Độ khó: **Medium** · Chủ đề: Math, Geometry (hình học toạ độ)

Skeleton LeetCode:

```python
class Solution:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho **4 điểm** trên mặt phẳng, **không theo thứ tự nào**, kiểm tra chúng có là 4 đỉnh của một **hình vuông** (4 cạnh bằng nhau và dương, 4 góc vuông) hay không.
2. Nắm các công cụ cần dùng: **bình phương khoảng cách** (tránh `sqrt`), **tích vô hướng** để kiểm tra vuông góc, `itertools.combinations` / `permutations`, `sorted`, `Counter`, và phép **cộng toạ độ** thay cho "trung điểm" để khỏi chia 2.
3. Đi từ **thử 24 thứ tự đỉnh** → **6 khoảng cách (4 cạnh bằng + 2 đường chéo bằng và gấp đôi)** → **mỗi bộ ba là tam giác vuông cân** → **hai đường chéo cắt nhau ở trung điểm, bằng nhau, vuông góc** → **sắp xếp điểm rồi chỉ thử một thứ tự**.
4. Hiểu mấu chốt: đề **không cho thứ tự** các điểm, nên ta cần một tiêu chí **không phụ thuộc thứ tự** — thường là *tập các khoảng cách* hay *tính chất đường chéo* — và mọi phép so sánh nên làm trên **số nguyên** (bình phương), không dùng số thực.
5. Tránh các lỗi kinh điển: so sánh số thực `sqrt`, giả định đầu vào đã theo thứ tự đỉnh, quên điều kiện cạnh **dương** (4 điểm trùng nhau), và chia nguyên khi tính trung điểm.

## 2. Đề bài & ví dụ

Cho toạ độ 4 điểm trong mặt phẳng `p1, p2, p3, p4`, mỗi điểm là `[x, y]`. Hãy trả về `true` nếu bốn điểm này tạo thành một **hình vuông**, ngược lại `false`.

**Hình vuông hợp lệ** có **bốn cạnh bằng nhau với độ dài dương** và **bốn góc bằng nhau (đều 90 độ)**. Các điểm được cho **không theo thứ tự** nào (có thể là thứ tự lộn xộn, ví dụ hai đỉnh đối diện đứng liền nhau).

**Ví dụ 1**

- Input: `p1 = [0,0], p2 = [1,1], p3 = [1,0], p4 = [0,1]`
- Output: `true`

**Ví dụ 2**

- Input: `p1 = [0,0], p2 = [1,1], p3 = [1,0], p4 = [0,12]`
- Output: `false`

**Ví dụ 3**

- Input: `p1 = [1,0], p2 = [-1,0], p3 = [0,1], p4 = [0,-1]`
- Output: `true` (hình vuông "xoay 45 độ", dạng hình thoi nằm nghiêng)

**Ràng buộc:**

- Mỗi `p_i` có đúng 2 phần tử `[x, y]`.
- `-10^4 <= x, y <= 10^4`.

Chú ý: bốn điểm có thể **trùng nhau** (ví dụ cả bốn đều là `[0,0]`) — khi đó cạnh có độ dài 0, **không** phải hình vuông.

In [1]:
from typing import List
from itertools import combinations, permutations, combinations_with_replacement, product
from collections import Counter
import math
import random
import time

examples = [
    ([0, 0], [1, 1], [1, 0], [0, 1], True),
    ([0, 0], [1, 1], [1, 0], [0, 12], False),
    ([1, 0], [-1, 0], [0, 1], [0, -1], True),
]
for *pts, expected in examples:
    print(pts, "-> kỳ vọng", expected)

[[0, 0], [1, 1], [1, 0], [0, 1]] -> kỳ vọng True
[[0, 0], [1, 1], [1, 0], [0, 12]] -> kỳ vọng False
[[1, 0], [-1, 0], [0, 1], [0, -1]] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Bình phương khoảng cách (squared distance)

Khoảng cách giữa `A = (x1, y1)` và `B = (x2, y2)` là `sqrt((x1-x2)² + (y1-y2)²)`. Để so sánh "hai cạnh có bằng nhau không", ta chỉ cần so **bình phương** khoảng cách: `(x1-x2)² + (y1-y2)²`. Đó là **số nguyên**, tính chính xác, và `sqrt` là hàm tăng nên `a = b ⇔ a² = b²` (với `a, b ≥ 0`). Với toạ độ tới `10^4`, bình phương khoảng cách tối đa `(2·10^4)² · 2 = 8·10^8`, Python không lo tràn số.

In [2]:
def sq_dist(p, q):
    return (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2

print(sq_dist([0, 0], [1, 1]))      # 2
print(sq_dist([0, 0], [3, 4]))      # 25 (khoảng cách thật là 5)
print(sq_dist([5, 5], [5, 5]))      # 0: hai điểm trùng nhau

2
25
0


### 3.2. Vector và tích vô hướng (dot product)

Vector từ `A` đến `B` là `AB = (xB - xA, yB - yA)`. **Tích vô hướng** của `u = (ux, uy)` và `v = (vx, vy)` là `u·v = ux·vx + uy·vy`. Hai vector (khác `0`) **vuông góc** khi và chỉ khi `u·v = 0`. Đây là cách kiểm tra góc vuông bằng toàn số nguyên.

In [3]:
def vec(a, b):
    return (b[0] - a[0], b[1] - a[1])

def dot(u, v):
    return u[0] * v[0] + u[1] * v[1]

print(dot(vec([0, 0], [1, 0]), vec([0, 0], [0, 1])))    # 0 -> vuông góc
print(dot(vec([0, 0], [1, 1]), vec([0, 0], [1, 0])))    # 1 -> không vuông
print(dot(vec([0, 0], [3, 4]), vec([0, 0], [-4, 3])))   # 0 -> vuông (xoay 90 độ: (x, y) -> (-y, x))

0
1
0


Phép **xoay 90 độ ngược chiều kim đồng hồ** quanh gốc: `(x, y) → (-y, x)`. Nó giữ nguyên độ dài và cho vector vuông góc với vector ban đầu. Ta sẽ dùng nó để **sinh hình vuông** làm test.

### 3.3. `combinations` và `permutations` của `itertools`

- `combinations(iterable, 2)`: mọi **cặp không thứ tự** — 4 điểm cho `C(4,2) = 6` cặp (đúng 6 khoảng cách).
- `combinations(iterable, 3)`: 4 bộ ba.
- `permutations(iterable)`: mọi **thứ tự** — 4 điểm cho `4! = 24` hoán vị.

In [4]:
pts = [[0, 0], [1, 1], [1, 0], [0, 1]]
print("số cặp   :", len(list(combinations(pts, 2))))
print("số bộ ba :", len(list(combinations(pts, 3))))
print("số hoán vị:", len(list(permutations(pts))))
for a, b in combinations(range(4), 2):
    print(f"cặp ({a},{b}): sq_dist = {sq_dist(pts[a], pts[b])}")

số cặp   : 6
số bộ ba : 4
số hoán vị: 24
cặp (0,1): sq_dist = 2
cặp (0,2): sq_dist = 1
cặp (0,3): sq_dist = 1
cặp (1,2): sq_dist = 1
cặp (1,3): sq_dist = 1
cặp (2,3): sq_dist = 2


### 3.4. `sorted`, `set`, `Counter`

- `sorted(list)` trả list đã sắp tăng dần (không đổi list gốc).
- `set(list)` giữ các giá trị **khác nhau**.
- `Counter(list)` đếm tần suất mỗi giá trị — rất hợp để nói "có đúng 4 khoảng cách bằng `d` và 2 khoảng cách bằng `2d`".

In [5]:
d = [1, 1, 1, 1, 2, 2]
print(sorted(d))
print(set(d))
print(Counter(d))
print(Counter([5, 5, 5, 5, 10, 10]))

[1, 1, 1, 1, 2, 2]
{1, 2}
Counter({1: 4, 2: 2})
Counter({5: 4, 10: 2})


### 3.5. "Trung điểm" mà không chia 2

Trung điểm của `A` và `C` là `((xA + xC) / 2, (yA + yC) / 2)`. Chia 2 sinh số lẻ (`.5`) hoặc nếu dùng `//` sẽ **làm tròn mất thông tin**. Để so sánh hai trung điểm có trùng nhau không, chỉ cần so **tổng toạ độ**: `xA + xC == xB + xD` và `yA + yC == yB + yD`. Bốn điểm `A, B, C, D` thỏa điều đó khi và chỉ khi hai đoạn `AC` và `BD` **cắt nhau tại trung điểm** — nghĩa là `ABCD` là hình bình hành.

In [6]:
A, B, C, D = [0, 0], [3, 1], [5, 4], [2, 3]          # hình bình hành ABCD
print("tổng A+C:", (A[0] + C[0], A[1] + C[1]), "| tổng B+D:", (B[0] + D[0], B[1] + D[1]))
print("hai trung điểm trùng nhau:", A[0] + C[0] == B[0] + D[0] and A[1] + C[1] == B[1] + D[1])

tổng A+C: (5, 4) | tổng B+D: (5, 4)
hai trung điểm trùng nhau: True


### 3.6. Mấy cách "định nghĩa" hình vuông tương đương

Một tứ giác `ABCD` là hình vuông khi (chọn một trong các cách):

1. 4 cạnh bằng nhau (dương) **và** 4 góc vuông; (thực ra cần 4 cạnh bằng + **một** góc vuông, hoặc 4 cạnh bằng + hai đường chéo bằng nhau).
2. Là hình bình hành (đường chéo cắt nhau ở trung điểm) **và** hai đường chéo **bằng nhau** (⇒ chữ nhật) **và** **vuông góc** (⇒ thoi) ⇒ vừa chữ nhật vừa thoi = hình vuông.
3. Mọi bộ ba đỉnh đều tạo **tam giác vuông cân**.

Mỗi định nghĩa gợi một thuật toán khác nhau. Ta sẽ dùng cả bốn (và một mẹo sắp xếp) ở mục 7.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: các điểm `(0,0), (1,1), (1,0), (0,1)`. Tính bình phương khoảng cách của **cả 6 cặp**:

| cặp | bình phương khoảng cách |
|---|---|
| `(0,0)-(1,1)` | `2` |
| `(0,0)-(1,0)` | `1` |
| `(0,0)-(0,1)` | `1` |
| `(1,1)-(1,0)` | `1` |
| `(1,1)-(0,1)` | `1` |
| `(1,0)-(0,1)` | `2` |

Có **4 giá trị `1`** (4 cạnh) và **2 giá trị `2`** (2 đường chéo). Đường chéo của hình vuông cạnh `a` dài `a·√2` nên bình phương gấp **2 lần** bình phương cạnh: `2 = 2 × 1` ✓.

Ví dụ 2: thay `(0,1)` bằng `(0,12)`: các khoảng cách bình phương gồm `2, 1, 144, 1, 122, 145`... không có 4 giá trị bằng nhau ⇒ không phải hình vuông.

Ví dụ 3: `(1,0), (-1,0), (0,1), (0,-1)`: bốn cạnh `sq = 2`, hai đường chéo `sq = 4 = 2 × 2` ✓ (hình vuông nghiêng).

**Trực giác chính:** hình vuông chỉ có **hai loại khoảng cách** giữa các cặp đỉnh — cạnh (4 lần) và đường chéo (2 lần) — và loại sau gấp đôi loại trước (ở dạng bình phương). Ta hãy xem bảng này bằng code.

In [7]:
def show_distances(pts):
    items = []
    for (i, a), (j, b) in combinations(enumerate(pts), 2):
        items.append((sq_dist(a, b), i, j))
    for d, i, j in sorted(items):
        print(f"  p{i + 1}{pts[i]} - p{j + 1}{pts[j]}: sq_dist = {d}")
    return sorted(d for d, _, _ in items)

for *pts, expected in examples:
    print(pts, "->", expected)
    print("  đã sắp:", show_distances(pts))

[[0, 0], [1, 1], [1, 0], [0, 1]] -> True
  p1[0, 0] - p3[1, 0]: sq_dist = 1
  p1[0, 0] - p4[0, 1]: sq_dist = 1
  p2[1, 1] - p3[1, 0]: sq_dist = 1
  p2[1, 1] - p4[0, 1]: sq_dist = 1
  p1[0, 0] - p2[1, 1]: sq_dist = 2
  p3[1, 0] - p4[0, 1]: sq_dist = 2
  đã sắp: [1, 1, 1, 1, 2, 2]
[[0, 0], [1, 1], [1, 0], [0, 12]] -> False
  p1[0, 0] - p3[1, 0]: sq_dist = 1
  p2[1, 1] - p3[1, 0]: sq_dist = 1
  p1[0, 0] - p2[1, 1]: sq_dist = 2
  p2[1, 1] - p4[0, 12]: sq_dist = 122
  p1[0, 0] - p4[0, 12]: sq_dist = 144
  p3[1, 0] - p4[0, 12]: sq_dist = 145
  đã sắp: [1, 1, 2, 122, 144, 145]
[[1, 0], [-1, 0], [0, 1], [0, -1]] -> True
  p1[1, 0] - p3[0, 1]: sq_dist = 2
  p1[1, 0] - p4[0, -1]: sq_dist = 2
  p2[-1, 0] - p3[0, 1]: sq_dist = 2
  p2[-1, 0] - p4[0, -1]: sq_dist = 2
  p1[1, 0] - p2[-1, 0]: sq_dist = 4
  p3[0, 1] - p4[0, -1]: sq_dist = 4
  đã sắp: [2, 2, 2, 2, 4, 4]


## 5. Cách 1 — Brute Force: thử mọi thứ tự đỉnh

**Ý tưởng:** nếu 4 điểm là hình vuông thì **có ít nhất một thứ tự** `A → B → C → D` đi quanh hình vuông. Ta thử **cả 24 hoán vị**. Với mỗi hoán vị, ta kiểm tra theo **định nghĩa**: bốn cạnh `AB, BC, CD, DA` bằng nhau và dương, và các góc tại `B, C, D` vuông (tích vô hướng hai cạnh kề bằng 0). Bốn cạnh bằng nhau cộng ba góc vuông thì góc còn lại tự vuông.

Rất chắc chắn đúng, bám sát đề, nên ta dùng làm **oracle** để kiểm tra các cách nhanh.

In [8]:
def valid_square_bruteforce(p1, p2, p3, p4):
    for a, b, c, d in permutations([p1, p2, p3, p4]):
        e = [vec(a, b), vec(b, c), vec(c, d), vec(d, a)]       # 4 cạnh dưới dạng vector
        lengths = [dot(v, v) for v in e]                       # bình phương độ dài
        if lengths[0] == 0 or len(set(lengths)) != 1:
            continue
        if dot(e[0], e[1]) == 0 and dot(e[1], e[2]) == 0 and dot(e[2], e[3]) == 0:
            return True
    return False

for *pts, expected in examples:
    got = valid_square_bruteforce(*pts)
    print(pts, got, "OK" if got == expected else "SAI")

[[0, 0], [1, 1], [1, 0], [0, 1]] True OK
[[0, 0], [1, 1], [1, 0], [0, 12]] False OK
[[1, 0], [-1, 0], [0, 1], [0, -1]] True OK


**Độ phức tạp:** `4! = 24` hoán vị, mỗi hoán vị tốn `O(1)` ⇒ `O(1)` (hằng số lớn khoảng vài trăm phép tính).

**Nút thắt:** không có nút thắt về độ lớn đầu vào (chỉ 4 điểm) — vấn đề là code **dài, nhiều chi tiết dễ sai** (thứ tự, vector, hoán vị) và lặp lại nhiều việc (một hình vuông xuất hiện trong 8 hoán vị: 4 điểm xuất phát × 2 chiều). Ta muốn một phép kiểm tra **không phụ thuộc thứ tự**.

In [9]:
good = [[0, 0], [1, 1], [1, 0], [0, 1]]
n_orders = 0
accepted = 0
for a, b, c, d in permutations(good):
    n_orders += 1
    e = [vec(a, b), vec(b, c), vec(c, d), vec(d, a)]
    L = [dot(v, v) for v in e]
    if L[0] > 0 and len(set(L)) == 1 and dot(e[0], e[1]) == 0 and dot(e[1], e[2]) == 0 and dot(e[2], e[3]) == 0:
        accepted += 1
print("trong", n_orders, "hoán vị, có", accepted, "hoán vị là một vòng quanh hình vuông (4 điểm xuất phát × 2 chiều)")

trong 24 hoán vị, có 8 hoán vị là một vòng quanh hình vuông (4 điểm xuất phát × 2 chiều)


## 6. Ý tưởng thuật toán / bước đột phá

Nhờ bảng ở mục 4, ta thấy một **chữ ký (signature)** của hình vuông không phụ thuộc thứ tự điểm: **6 bình phương khoảng cách** có dạng `d, d, d, d, 2d, 2d` với `d > 0`.

Cụ thể, gọi `s` = 6 giá trị đã **sắp tăng dần**; hình vuông khi và chỉ khi:

- `s[0] > 0` (cạnh dương, tức các điểm không trùng nhau),
- `s[0] == s[1] == s[2] == s[3]` (4 cạnh bằng nhau; vì `d` là giá trị nhỏ nhất),
- `s[4] == s[5]` (2 đường chéo bằng nhau),
- `s[4] == 2 * s[0]` (đường chéo² gấp đôi cạnh²).

Vì sao chỉ **cần đủ** như vậy (không sót trường hợp "giả")? Ta sẽ vừa chứng minh vừa kiểm tra bằng code ở mục 10. Điểm chính: nếu 4 khoảng cách nhỏ nhất bằng nhau thì có 4 cặp tạo "tứ giác đều cạnh" (thoi hoặc suy biến), và đường chéo thỏa `diag² = 2·cạnh²` (định lý Pythagoras đảo) cho góc vuông.

Cách này chỉ cần **tính 6 số và sắp xếp** — `O(1)`, ngắn gọn, không cần thử thứ tự.

## 7. Các cách tiếp theo

Cùng một bài, ta có thể thấy hình vuông qua nhiều "lăng kính": **chữ ký 6 khoảng cách**, **bộ ba tam giác vuông cân**, **đường chéo**, và **sắp xếp để chỉ còn một thứ tự**. Khi nộp LeetCode, chỉ đổi tên class thành `Solution`.

### 7.1. Cách 2 — Chữ ký 6 khoảng cách

**Giải thích từng đoạn:**

1. Lấy 4 điểm vào list `pts`.
2. Với mỗi trong 6 cặp, tính `sq_dist`; sắp tăng dần được `d[0..5]`.
3. Kiểm tra `d[0] > 0`, `d[0] == d[3]`, `d[4] == d[5]`, `d[4] == 2 * d[0]`.

In [10]:
class SolutionDistances:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        def dist(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

        d = sorted([dist(p1, p2), dist(p1, p3), dist(p1, p4),
                    dist(p2, p3), dist(p2, p4), dist(p3, p4)])
        return d[0] > 0 and d[0] == d[3] and d[4] == d[5] and d[4] == 2 * d[0]

for *pts, expected in examples:
    print(pts, SolutionDistances().validSquare(*pts), "kỳ vọng", expected)

[[0, 0], [1, 1], [1, 0], [0, 1]] True kỳ vọng True
[[0, 0], [1, 1], [1, 0], [0, 12]] False kỳ vọng False
[[1, 0], [-1, 0], [0, 1], [0, -1]] True kỳ vọng True


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        def dist(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

        d = sorted([dist(p1, p2), dist(p1, p3), dist(p1, p4),
                    dist(p2, p3), dist(p2, p4), dist(p3, p4)])
        return d[0] > 0 and d[0] == d[3] and d[4] == d[5] and d[4] == 2 * d[0]
```

**Độ phức tạp:** thời gian `O(1)` (6 khoảng cách + sắp xếp 6 phần tử), bộ nhớ `O(1)`.

### 7.2. Cách 3 — Mọi bộ ba là tam giác vuông cân

**Ý tưởng:** trong hình vuông, bất kỳ 3 đỉnh nào cũng tạo thành **tam giác vuông cân** (đỉnh vuông ở đỉnh giữa). Vậy ta xét cả `C(4,3) = 4` bộ ba; mỗi bộ ba có 3 cạnh bình phương `a ≤ b ≤ c`, cần: `a > 0`, `a == b` (cân), `a + b == c` (Pythagoras: vuông). Nếu cả 4 bộ ba đạt thì là hình vuông.

**Giải thích từng đoạn:**

1. Với mỗi bộ ba `(x, y, z)` tính 3 bình phương cạnh, sắp tăng dần được `a, b, c`.
2. Hàm `is_isosceles_right` trả `True` nếu `a > 0 and a == b and a + b == c`.
3. Trả về `True` nếu cả 4 bộ ba đều đạt.

In [11]:
class SolutionTriples:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        def dist(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

        def is_isosceles_right(x, y, z):
            a, b, c = sorted([dist(x, y), dist(y, z), dist(x, z)])
            return a > 0 and a == b and a + b == c

        return (is_isosceles_right(p1, p2, p3) and is_isosceles_right(p1, p2, p4)
                and is_isosceles_right(p1, p3, p4) and is_isosceles_right(p2, p3, p4))

for *pts, expected in examples:
    print(pts, SolutionTriples().validSquare(*pts), "kỳ vọng", expected)

[[0, 0], [1, 1], [1, 0], [0, 1]] True kỳ vọng True
[[0, 0], [1, 1], [1, 0], [0, 12]] False kỳ vọng False
[[1, 0], [-1, 0], [0, 1], [0, -1]] True kỳ vọng True


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        def dist(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

        def is_isosceles_right(x, y, z):
            a, b, c = sorted([dist(x, y), dist(y, z), dist(x, z)])
            return a > 0 and a == b and a + b == c

        return (is_isosceles_right(p1, p2, p3) and is_isosceles_right(p1, p2, p4)
                and is_isosceles_right(p1, p3, p4) and is_isosceles_right(p2, p3, p4))
```

**Độ phức tạp:** `O(1)` thời gian và bộ nhớ (4 bộ ba × 3 cạnh).

### 7.3. Cách 4 — Hai đường chéo: cắt nhau ở trung điểm, bằng nhau, vuông góc

**Ý tưởng:** hình vuông là hình bình hành + hai đường chéo bằng nhau + hai đường chéo vuông góc. Ta **không biết** cặp điểm nào là đường chéo, nhưng đường chéo luôn nối `p1` với **một trong ba** điểm còn lại. Thử cả 3:

1. Chọn `p1` và `q` là hai đầu một đường chéo; hai điểm còn lại `r, s` là hai đầu đường chéo kia.
2. **Cùng trung điểm:** `p1 + q == r + s` (cộng toạ độ, không chia 2).
3. **Bằng nhau và dương:** `|p1 q|² == |r s|² > 0`.
4. **Vuông góc:** `(q - p1) · (s - r) == 0`.
5. Nếu một cách chọn thỏa thì đáp án `True`.

(Nếu hai đường chéo cắt nhau tại trung điểm, bằng nhau, vuông góc và dương thì tứ giác có 4 cạnh bằng nhau và 4 góc vuông.)

In [12]:
class SolutionDiagonals:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        pts = [p1, p2, p3, p4]
        for k in (1, 2, 3):
            q = pts[k]
            r, s = [pts[i] for i in (1, 2, 3) if i != k]
            same_mid = p1[0] + q[0] == r[0] + s[0] and p1[1] + q[1] == r[1] + s[1]
            u = (q[0] - p1[0], q[1] - p1[1])          # đường chéo p1-q
            v = (s[0] - r[0], s[1] - r[1])            # đường chéo r-s
            equal = u[0] * u[0] + u[1] * u[1] == v[0] * v[0] + v[1] * v[1] > 0
            perpendicular = u[0] * v[0] + u[1] * v[1] == 0
            if same_mid and equal and perpendicular:
                return True
        return False

for *pts, expected in examples:
    print(pts, SolutionDiagonals().validSquare(*pts), "kỳ vọng", expected)

[[0, 0], [1, 1], [1, 0], [0, 1]] True kỳ vọng True
[[0, 0], [1, 1], [1, 0], [0, 12]] False kỳ vọng False
[[1, 0], [-1, 0], [0, 1], [0, -1]] True kỳ vọng True


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        pts = [p1, p2, p3, p4]
        for k in (1, 2, 3):
            q = pts[k]
            r, s = [pts[i] for i in (1, 2, 3) if i != k]
            same_mid = p1[0] + q[0] == r[0] + s[0] and p1[1] + q[1] == r[1] + s[1]
            u = (q[0] - p1[0], q[1] - p1[1])
            v = (s[0] - r[0], s[1] - r[1])
            equal = u[0] * u[0] + u[1] * u[1] == v[0] * v[0] + v[1] * v[1] > 0
            perpendicular = u[0] * v[0] + u[1] * v[1] == 0
            if same_mid and equal and perpendicular:
                return True
        return False
```

**Độ phức tạp:** `O(1)` thời gian (3 lần thử), `O(1)` bộ nhớ.

### 7.4. Cách 5 — Sắp xếp các điểm rồi chỉ thử một thứ tự

**Ý tưởng:** sắp 4 điểm theo toạ độ (`x` rồi `y`) được `a, b, c, d`. Nếu bốn điểm là hình vuông thì điểm nhỏ nhất `a` và lớn nhất `d` (theo thứ tự từ điển) **luôn là hai đỉnh đối diện**, nên vòng quanh hình vuông là `a → b → d → c`. Vậy chỉ cần **kiểm tra duy nhất** thứ tự `a, b, d, c`: bốn cạnh bằng nhau, dương, và các góc vuông.

(Lý do `a` và `d` đối diện: `a` là đỉnh "trái nhất" (và thấp nhất khi trùng `x`), `d` là đỉnh "phải nhất"; trong hình vuông, đỉnh trái nhất và đỉnh phải nhất luôn là hai đỉnh đối nhau, kể cả khi hình vuông nằm thẳng. Ta sẽ kiểm bằng code vét cạn ở mục 12.)

In [13]:
class SolutionSorted:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        a, b, c, d = sorted([p1, p2, p3, p4])
        order = [a, b, d, c]                              # vòng quanh: a -> b -> d -> c -> a
        edges = [(order[(i + 1) % 4][0] - order[i][0], order[(i + 1) % 4][1] - order[i][1]) for i in range(4)]
        lens = [e[0] * e[0] + e[1] * e[1] for e in edges]
        if lens[0] == 0 or len(set(lens)) != 1:
            return False
        return all(edges[i][0] * edges[(i + 1) % 4][0] + edges[i][1] * edges[(i + 1) % 4][1] == 0
                   for i in range(4))

for *pts, expected in examples:
    print(pts, SolutionSorted().validSquare(*pts), "kỳ vọng", expected)

[[0, 0], [1, 1], [1, 0], [0, 1]] True kỳ vọng True
[[0, 0], [1, 1], [1, 0], [0, 12]] False kỳ vọng False
[[1, 0], [-1, 0], [0, 1], [0, -1]] True kỳ vọng True


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        a, b, c, d = sorted([p1, p2, p3, p4])
        order = [a, b, d, c]
        edges = [(order[(i + 1) % 4][0] - order[i][0], order[(i + 1) % 4][1] - order[i][1]) for i in range(4)]
        lens = [e[0] * e[0] + e[1] * e[1] for e in edges]
        if lens[0] == 0 or len(set(lens)) != 1:
            return False
        return all(edges[i][0] * edges[(i + 1) % 4][0] + edges[i][1] * edges[(i + 1) % 4][1] == 0
                   for i in range(4))
```

**Độ phức tạp:** `O(1)` (sắp 4 phần tử + 4 cạnh), bộ nhớ `O(1)`.

Cách này khác cách 1 ở chỗ: thay vì thử 24 thứ tự, ta **ép** thứ tự duy nhất có thể đúng. Nếu 4 điểm không phải hình vuông thì không có thứ tự nào thỏa nên kết quả vẫn `False`.

## 8. TRACE từng bước

Ta theo dõi cách 2 trên ví dụ 3 `(1,0), (-1,0), (0,1), (0,-1)` và ví dụ 2 `(0,0), (1,1), (1,0), (0,12)`: in 6 cặp, danh sách sau khi sắp, và từng điều kiện.

Bảng trạng thái cho ví dụ 3:

| cặp | sq_dist |
|---|---|
| `p1-p2` `(1,0)-(-1,0)` | 4 |
| `p1-p3` `(1,0)-(0,1)` | 2 |
| `p1-p4` `(1,0)-(0,-1)` | 2 |
| `p2-p3` `(-1,0)-(0,1)` | 2 |
| `p2-p4` `(-1,0)-(0,-1)` | 2 |
| `p3-p4` `(0,1)-(0,-1)` | 4 |

Sắp: `[2, 2, 2, 2, 4, 4]` → `d[0] = 2 > 0`, `d[0] == d[3]`, `d[4] == d[5] = 4`, `4 == 2·2` ⇒ `True`.

In [14]:
def trace_distances(p1, p2, p3, p4):
    pts = [p1, p2, p3, p4]
    print("điểm:", pts)
    ds = []
    for i, j in combinations(range(4), 2):
        d = sq_dist(pts[i], pts[j])
        ds.append(d)
        print(f"  cặp p{i + 1}-p{j + 1}: {pts[i]} - {pts[j]} -> sq_dist = {d}")
    d = sorted(ds)
    print("đã sắp:", d)
    c1 = d[0] > 0
    c2 = d[0] == d[3]
    c3 = d[4] == d[5]
    c4 = d[4] == 2 * d[0]
    print(f"  d[0] > 0            : {d[0]} > 0 -> {c1}")
    print(f"  d[0] == d[3]        : {d[0]} == {d[3]} -> {c2}")
    print(f"  d[4] == d[5]        : {d[4]} == {d[5]} -> {c3}")
    print(f"  d[4] == 2 * d[0]    : {d[4]} == {2 * d[0]} -> {c4}")
    return c1 and c2 and c3 and c4

print("KẾT QUẢ ví dụ 3:", trace_distances([1, 0], [-1, 0], [0, 1], [0, -1]))
print()
print("KẾT QUẢ ví dụ 2:", trace_distances([0, 0], [1, 1], [1, 0], [0, 12]))

điểm: [[1, 0], [-1, 0], [0, 1], [0, -1]]
  cặp p1-p2: [1, 0] - [-1, 0] -> sq_dist = 4
  cặp p1-p3: [1, 0] - [0, 1] -> sq_dist = 2
  cặp p1-p4: [1, 0] - [0, -1] -> sq_dist = 2
  cặp p2-p3: [-1, 0] - [0, 1] -> sq_dist = 2
  cặp p2-p4: [-1, 0] - [0, -1] -> sq_dist = 2
  cặp p3-p4: [0, 1] - [0, -1] -> sq_dist = 4
đã sắp: [2, 2, 2, 2, 4, 4]
  d[0] > 0            : 2 > 0 -> True
  d[0] == d[3]        : 2 == 2 -> True
  d[4] == d[5]        : 4 == 4 -> True
  d[4] == 2 * d[0]    : 4 == 4 -> True
KẾT QUẢ ví dụ 3: True

điểm: [[0, 0], [1, 1], [1, 0], [0, 12]]
  cặp p1-p2: [0, 0] - [1, 1] -> sq_dist = 2
  cặp p1-p3: [0, 0] - [1, 0] -> sq_dist = 1
  cặp p1-p4: [0, 0] - [0, 12] -> sq_dist = 144
  cặp p2-p3: [1, 1] - [1, 0] -> sq_dist = 1
  cặp p2-p4: [1, 1] - [0, 12] -> sq_dist = 122
  cặp p3-p4: [1, 0] - [0, 12] -> sq_dist = 145
đã sắp: [1, 1, 2, 122, 144, 145]
  d[0] > 0            : 1 > 0 -> True
  d[0] == d[3]        : 1 == 122 -> False
  d[4] == d[5]        : 144 == 145 -> False
  d[4] == 2 * 

Giờ trace cách 4 (đường chéo): với mỗi cách chọn đối tác của `p1`, in tổng toạ độ (trung điểm), độ dài hai đường chéo, và tích vô hướng.

In [15]:
def trace_diagonals(p1, p2, p3, p4):
    pts = [p1, p2, p3, p4]
    for k in (1, 2, 3):
        q = pts[k]
        r, s = [pts[i] for i in (1, 2, 3) if i != k]
        mid1 = (p1[0] + q[0], p1[1] + q[1])
        mid2 = (r[0] + s[0], r[1] + s[1])
        u = vec(p1, q)
        v = vec(r, s)
        print(f"thử p1 đối với p{k + 1}: đường chéo {p1}-{q} và {r}-{s}")
        print(f"   tổng toạ độ: {mid1} vs {mid2} -> cùng trung điểm: {mid1 == mid2}")
        print(f"   |u|² = {dot(u, u)}, |v|² = {dot(v, v)} -> bằng nhau và dương: {dot(u, u) == dot(v, v) > 0}")
        print(f"   u·v = {dot(u, v)} -> vuông góc: {dot(u, v) == 0}")
        if mid1 == mid2 and dot(u, u) == dot(v, v) > 0 and dot(u, v) == 0:
            return True
    return False

print("KẾT QUẢ:", trace_diagonals([0, 0], [1, 1], [1, 0], [0, 1]))

thử p1 đối với p2: đường chéo [0, 0]-[1, 1] và [1, 0]-[0, 1]
   tổng toạ độ: (1, 1) vs (1, 1) -> cùng trung điểm: True
   |u|² = 2, |v|² = 2 -> bằng nhau và dương: True
   u·v = 0 -> vuông góc: True
KẾT QUẢ: True


## 9. Code nộp LeetCode

Ta chọn **Cách 2 (chữ ký 6 khoảng cách)**: ngắn nhất, dễ nhớ, không phụ thuộc thứ tự điểm. Class `Solution` đúng chữ ký:

```python
class Solution:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        def dist(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

        d = sorted([dist(p1, p2), dist(p1, p3), dist(p1, p4),
                    dist(p2, p3), dist(p2, p4), dist(p3, p4)])
        return d[0] > 0 and d[0] == d[3] and d[4] == d[5] and d[4] == 2 * d[0]
```

In [16]:
class Solution:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        def dist(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

        d = sorted([dist(p1, p2), dist(p1, p3), dist(p1, p4),
                    dist(p2, p3), dist(p2, p4), dist(p3, p4)])
        return d[0] > 0 and d[0] == d[3] and d[4] == d[5] and d[4] == 2 * d[0]

for *pts, expected in examples:
    got = Solution().validSquare(*pts)
    assert got == expected, (pts, got, expected)
    print(pts, "->", got, "OK")

[[0, 0], [1, 1], [1, 0], [0, 1]] -> True OK
[[0, 0], [1, 1], [1, 0], [0, 12]] -> False OK
[[1, 0], [-1, 0], [0, 1], [0, -1]] -> True OK


## 10. Vì sao đúng

**Chiều thuận (hình vuông ⇒ chữ ký).** Hình vuông cạnh `a`: có 4 cặp kề nhau cách `a²`, 2 cặp đối diện (đường chéo) cách `(a√2)² = 2a²`. Sắp tăng: `a, a, a, a, 2a², 2a²` (với `a > 0`): thỏa cả 4 điều kiện.

**Chiều đảo (chữ ký ⇒ hình vuông).** Giả sử 6 khoảng cách² sắp được `d, d, d, d, 2d, 2d` với `d > 0`. Gọi 4 cặp có khoảng cách² bằng `d` là "cặp ngắn" và 2 cặp còn lại (cách² `2d`) là "cặp dài". Hai cặp dài trong 4 điểm chỉ có thể xảy ra theo hai kiểu:

- **Hai cặp dài chung một đỉnh**, chẳng hạn `AB` và `AC` (cách² `2d`), còn `BC, AD, BD, CD` đều cách² `d`. Khi đó `D` cách đều `A, B, C` một khoảng `√d`, tức `D` là tâm đường tròn ngoại tiếp tam giác `ABC` có ba cạnh² là `2d, 2d, d`. Công thức Heron cho `16·S² = 7d²`, nên bán kính ngoại tiếp `R = abc / (4S)` có `R² = 4d/7`, **khác `d`** ⇒ mâu thuẫn. Kiểu này không xảy ra.
- **Hai cặp dài rời nhau**, chẳng hạn `AC` và `BD`, còn `AB, BC, CD, DA` đều cách² `d`. Đó là tứ giác `ABCD` có 4 cạnh bằng nhau (dương). Vì `AB² + BC² = d + d = 2d = AC²`, theo **đảo định lý Pythagoras** góc tại `B` vuông; tương tự góc tại `C` (tam giác `BCD`) và góc tại `D` (tam giác `CDA`) vuông. Bốn cạnh bằng nhau và ba góc vuông ⇒ **hình vuông**.

Vậy chữ ký `d,d,d,d,2d,2d` (`d > 0`) là điều kiện **cần và đủ**. Ngoài lập luận, ta còn **kiểm tra bằng máy**: so cách 2 với brute force trên **mọi bộ 4 điểm** của lưới nhỏ (kể cả điểm trùng nhau).

In [17]:
grid = [[x, y] for x in range(-2, 3) for y in range(-2, 3)]          # lưới 5 x 5 = 25 điểm
total = squares = 0
mismatch = 0
for quad in combinations_with_replacement(grid, 4):                 # mọi đa tập 4 điểm (kể cả trùng)
    a = valid_square_bruteforce(*quad)
    b = Solution().validSquare(*quad)
    total += 1
    squares += a
    if a != b:
        mismatch += 1
print("số bộ 4 điểm (đa tập) đã kiểm:", total, "| trong đó là hình vuông:", squares, "| sai lệch:", mismatch)
assert mismatch == 0

số bộ 4 điểm (đa tập) đã kiểm: 20475 | trong đó là hình vuông: 50 | sai lệch: 0


Không có sai lệch trên toàn bộ lưới 5×5 (kể cả bộ có điểm trùng), khớp với chứng minh ở trên.

Với **Cách 3** (bộ ba), **Cách 4** (đường chéo) và **Cách 5** (sắp xếp), ta cũng đối chiếu đầy đủ với oracle ở mục 12.

## 11. Time / Space Complexity

Chỉ có **4 điểm** nên mọi cách đều là `O(1)`; điều khác nhau là hằng số và độ đơn giản.

| Cách | Số phép chính | Time | Space |
|---|---|---|---|
| 1. 24 hoán vị | 24 vòng, mỗi vòng ~ 4 cạnh + 3 tích vô hướng | `O(1)` (hằng lớn) | `O(1)` |
| 2. 6 khoảng cách | 6 khoảng cách + sắp 6 số | `O(1)` | `O(1)` |
| 3. 4 bộ ba | 4 × (3 khoảng cách + sắp 3 số) | `O(1)` | `O(1)` |
| 4. Đường chéo | 3 lần thử × vài phép | `O(1)` | `O(1)` |
| 5. Sắp xếp rồi 1 thứ tự | sắp 4 điểm + 4 cạnh | `O(1)` | `O(1)` |

Nếu tổng quát hóa cho `n` điểm (ví dụ đếm số hình vuông, mục 15), hằng số trở thành `O(n⁴)` cho brute force hay `O(n²)` cho cách băm. Ta đo thời gian thực tế của từng cách trên 20 000 lần gọi để thấy tương quan.

In [18]:
rng = random.Random(11)
samples = []
for _ in range(20000):
    samples.append([[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(4)])

impls = {
    "1. hoán vị (24)": valid_square_bruteforce,
    "2. 6 khoảng cách": lambda *p: Solution().validSquare(*p),
    "3. bộ ba": lambda *p: SolutionTriples().validSquare(*p),
    "4. đường chéo": lambda *p: SolutionDiagonals().validSquare(*p),
    "5. sắp xếp": lambda *p: SolutionSorted().validSquare(*p),
}
for name, fn in impls.items():
    t0 = time.perf_counter()
    cnt = sum(fn(*q) for q in samples)
    dt = time.perf_counter() - t0
    print(f"{name:18s} {dt:6.3f}s cho {len(samples)} lần gọi ({cnt} hình vuông)")

1. hoán vị (24)     0.447s cho 20000 lần gọi (3 hình vuông)
2. 6 khoảng cách    0.024s cho 20000 lần gọi (3 hình vuông)
3. bộ ba            0.018s cho 20000 lần gọi (3 hình vuông)
4. đường chéo       0.036s cho 20000 lần gọi (3 hình vuông)
5. sắp xếp          0.039s cho 20000 lần gọi (3 hình vuông)


## 12. Bộ test

Gom các cách vào một dict để so sánh với nhau và với oracle. Các phần: ví dụ đề, edge case, phản ví dụ (những hình "gần giống" vuông), vét cạn toàn lưới, ngẫu nhiên có seed (sinh hình vuông thật bằng phép xoay vector), và ràng buộc biên `±10^4`.

In [19]:
methods = {
    "perm": lambda p: valid_square_bruteforce(*p),
    "dist": lambda p: SolutionDistances().validSquare(*p),
    "triples": lambda p: SolutionTriples().validSquare(*p),
    "diagonals": lambda p: SolutionDiagonals().validSquare(*p),
    "sorted": lambda p: SolutionSorted().validSquare(*p),
    "final": lambda p: Solution().validSquare(*p),
}

def check_all(pts, expected=None):
    results = {name: fn(pts) for name, fn in methods.items()}
    assert len(set(results.values())) == 1, (pts, results)
    r = results["final"]
    if expected is not None:
        assert r == expected, (pts, r, expected)
    return r

# 1) Ví dụ đề
for *pts, expected in examples:
    check_all(pts, expected)
print("ví dụ đề: OK")

# 2) Edge case
edge = [
    ([[0, 0], [0, 0], [0, 0], [0, 0]], False),          # 4 điểm trùng nhau
    ([[0, 0], [0, 0], [1, 0], [1, 0]], False),          # trùng từng đôi
    ([[0, 0], [1, 0], [2, 0], [3, 0]], False),          # thẳng hàng
    ([[0, 0], [2, 0], [2, 2], [0, 2]], True),           # vuông thẳng, đúng thứ tự
    ([[0, 0], [2, 2], [2, 0], [0, 2]], True),           # vuông thẳng, thứ tự lộn xộn (đường chéo đứng liền nhau)
    ([[0, 0], [2, 1], [1, 3], [-1, 2]], True),          # vuông nghiêng cạnh (2,1)
    ([[0, 0], [3, 0], [3, 1], [0, 1]], False),          # chữ nhật 3 x 1
    ([[0, 0], [3, 4], [8, 4], [5, 0]], False),          # hình thoi cạnh 5 (4 cạnh bằng nhau nhưng không vuông)
    ([[0, 0], [1, 0], [0, 1], [5, 5]], False),          # 3 điểm tạo góc vuông + điểm xa
    ([[0, 0], [1, 0], [0, 1], [1, 1]], True),
    ([[10000, 10000], [-10000, 10000], [-10000, -10000], [10000, -10000]], True),   # cạnh 20000 (biên)
    ([[10000, 0], [0, 10000], [-10000, 0], [0, -10000]], True),                     # nghiêng, biên
]
for pts, expected in edge:
    check_all(pts, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


In [20]:
# 3) Phản ví dụ: các hình "gần" vuông mà nhiều cách hiểu sai sẽ nhận nhầm
near_miss = {
    "hình thoi (4 cạnh bằng)": [[0, 0], [3, 4], [8, 4], [5, 0]],
    "chữ nhật (4 góc vuông)": [[0, 0], [4, 0], [4, 3], [0, 3]],
    "tam giác vuông cân + tâm": [[0, 0], [2, 0], [0, 2], [1, 1]],
    "hình bình hành": [[0, 0], [4, 0], [5, 3], [1, 3]],
    "cân đối nhưng lệch một điểm": [[0, 0], [2, 0], [2, 2], [0, 3]],
}
for name, pts in near_miss.items():
    r = check_all(pts)
    print(f"{name:30s} {pts} -> {r}")
    assert r is False

# 4) Vét cạn toàn lưới 4 x 4 (đa tập 4 điểm, mỗi bộ đảo thứ tự ngẫu nhiên): mọi cách khớp oracle
rng = random.Random(593)
grid = [[x, y] for x in range(4) for y in range(4)]
n_total = n_true = 0
for quad in combinations_with_replacement(grid, 4):
    q = list(quad)
    rng.shuffle(q)
    r = check_all(q)
    n_total += 1
    n_true += r
print("lưới 4x4:", n_total, "bộ 4 điểm,", n_true, "hình vuông, 5 cách đều khớp oracle: OK")

hình thoi (4 cạnh bằng)        [[0, 0], [3, 4], [8, 4], [5, 0]] -> False
chữ nhật (4 góc vuông)         [[0, 0], [4, 0], [4, 3], [0, 3]] -> False
tam giác vuông cân + tâm       [[0, 0], [2, 0], [0, 2], [1, 1]] -> False
hình bình hành                 [[0, 0], [4, 0], [5, 3], [1, 3]] -> False
cân đối nhưng lệch một điểm    [[0, 0], [2, 0], [2, 2], [0, 3]] -> False


lưới 4x4: 3876 bộ 4 điểm, 20 hình vuông, 5 cách đều khớp oracle: OK


In [21]:
# 5) Ngẫu nhiên có seed: sinh hình vuông thật bằng xoay vector, rồi xáo trộn; và bản "bị bóp méo" 1 điểm
def random_square(rng, limit=10000):
    while True:
        a, b = rng.randint(-3000, 3000), rng.randint(-3000, 3000)
        if (a, b) == (0, 0):
            continue
        ox, oy = rng.randint(-limit // 3, limit // 3), rng.randint(-limit // 3, limit // 3)
        v = (a, b)
        w = (-b, a)                                         # xoay 90 độ
        pts = [[ox, oy], [ox + v[0], oy + v[1]], [ox + v[0] + w[0], oy + v[1] + w[1]], [ox + w[0], oy + w[1]]]
        if all(-limit <= c <= limit for p in pts for c in p):
            return pts

rng = random.Random(2026)
n_sq = n_bad = 0
for _ in range(3000):
    sq = random_square(rng)
    rng.shuffle(sq)
    assert check_all(sq) is True
    n_sq += 1
    bad = [p[:] for p in sq]
    k = rng.randrange(4)
    axis = rng.randrange(2)
    bad[k][axis] += rng.choice([-2, -1, 1, 2])
    if all(-10000 <= c <= 10000 for p in bad for c in p):
        expected = valid_square_bruteforce(*bad)             # oracle quyết định
        assert check_all(bad) == expected
        n_bad += 1
print("hình vuông thật:", n_sq, "| bản bị dịch chuyển 1 điểm:", n_bad, "-> mọi cách khớp: OK")

# 6) Ngẫu nhiên hoàn toàn trong miền rộng + hay trùng điểm (miền hẹp)
for _ in range(3000):
    lim = rng.choice([1, 2, 5, 50, 10000])
    pts = [[rng.randint(-lim, lim), rng.randint(-lim, lim)] for _ in range(4)]
    assert check_all(pts) == valid_square_bruteforce(*pts)
print("ngẫu nhiên hoàn toàn: OK")

hình vuông thật: 3000 | bản bị dịch chuyển 1 điểm: 3000 -> mọi cách khớp: OK


ngẫu nhiên hoàn toàn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: so sánh số thực (`sqrt`) bằng `==`

Tính khoảng cách thật bằng `math.sqrt`, rồi kiểm `đường chéo == cạnh * sqrt(2)`: sai số dấu phẩy động làm hình vuông **thật** bị loại.

In [22]:
class WrongFloat:
    def validSquare(self, p1, p2, p3, p4):
        ds = sorted(math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)
                    for a, b in combinations([p1, p2, p3, p4], 2))
        return ds[0] > 0 and ds[0] == ds[3] and ds[4] == ds[5] and ds[4] == ds[0] * math.sqrt(2)

sq = [[0, 0], [3843, 2020], [1823, 5863], [-2020, 3843]]     # hình vuông thật (cạnh vector (3843, 2020))
print("sai :", WrongFloat().validSquare(*sq), "(đúng phải là True)")
print("đúng:", Solution().validSquare(*sq))

sai : False (đúng phải là True)
đúng: True


**Vì sao sai:** `sqrt` trả số thực gần đúng; `cạnh * sqrt(2)` và `sqrt(2 * cạnh²)` có thể lệch nhau ở chữ số cuối, nên phép `==` cho `False`. **Cách sửa:** chỉ so sánh **bình phương khoảng cách** (số nguyên): `d[4] == 2 * d[0]`.

### Lỗi 2: giả định đầu vào đã theo thứ tự các đỉnh

Đề nói rõ "không theo thứ tự nào". Nếu coi `p1 → p2 → p3 → p4` là một vòng quanh hình vuông thì ví dụ 1 sẽ sai.

In [23]:
class WrongOrder:
    def validSquare(self, p1, p2, p3, p4):
        ring = [p1, p2, p3, p4]
        edges = [vec(ring[i], ring[(i + 1) % 4]) for i in range(4)]
        lens = [dot(e, e) for e in edges]
        return (lens[0] > 0 and len(set(lens)) == 1 and
                all(dot(edges[i], edges[(i + 1) % 4]) == 0 for i in range(4)))

print("sai :", WrongOrder().validSquare([0, 0], [1, 1], [1, 0], [0, 1]), "(ví dụ 1: đúng phải là True)")
print("đúng:", Solution().validSquare([0, 0], [1, 1], [1, 0], [0, 1]))

sai : False (ví dụ 1: đúng phải là True)
đúng: True


**Vì sao sai:** trong ví dụ 1 `p1 = (0,0)` và `p2 = (1,1)` là **hai đỉnh đối diện** (đường chéo), nhưng code coi chúng liền kề. **Cách sửa:** dùng tiêu chí không phụ thuộc thứ tự (tập khoảng cách) hoặc thử mọi thứ tự / sắp xếp trước.

### Lỗi 3: quên điều kiện cạnh **dương**

In [24]:
class WrongPositive:
    def validSquare(self, p1, p2, p3, p4):
        d = sorted(sq_dist(a, b) for a, b in combinations([p1, p2, p3, p4], 2))
        return d[0] == d[3] and d[4] == d[5] and d[4] == 2 * d[0]          # thiếu d[0] > 0

print("sai :", WrongPositive().validSquare([0, 0], [0, 0], [0, 0], [0, 0]), "(4 điểm trùng nhau: đúng phải là False)")
print("đúng:", Solution().validSquare([0, 0], [0, 0], [0, 0], [0, 0]))

sai : True (4 điểm trùng nhau: đúng phải là False)
đúng: False


**Vì sao sai:** bốn điểm trùng nhau cho 6 khoảng cách đều `0`; mọi điều kiện `0 == 0` thỏa. **Cách sửa:** thêm `d[0] > 0` (cạnh phải có độ dài dương, đúng như đề yêu cầu).

### Lỗi 4: chia nguyên khi so sánh trung điểm

In [25]:
class WrongMidpoint:
    def validSquare(self, p1, p2, p3, p4):
        pts = [p1, p2, p3, p4]
        for k in (1, 2, 3):
            q = pts[k]
            r, s = [pts[i] for i in (1, 2, 3) if i != k]
            same_mid = (p1[0] + q[0]) // 2 == (r[0] + s[0]) // 2 and (p1[1] + q[1]) // 2 == (r[1] + s[1]) // 2   # SAI
            u, v = vec(p1, q), vec(r, s)
            if same_mid and dot(u, u) == dot(v, v) > 0 and dot(u, v) == 0:
                return True
        return False

bad = [[2, 0], [3, 4], [0, 3], [4, 2]]
print("sai :", WrongMidpoint().validSquare(*bad), "(đúng phải là False)")
print("đúng:", Solution().validSquare(*bad), "| oracle:", valid_square_bruteforce(*bad))

sai : True (đúng phải là False)
đúng: False | oracle: False


**Vì sao sai:** `//` làm tròn xuống nên tổng `5` và `4` đều cho `2`: hai trung điểm khác nhau bị coi là trùng. **Cách sửa:** so sánh **tổng** toạ độ, không chia.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài hình học toạ độ nguyên:**

- Bài cho **điểm nguyên** và hỏi "có tạo hình X không" ⇒ làm việc với **bình phương khoảng cách**, **tích vô hướng** (vuông góc), **tích có hướng** (cùng phía, thẳng hàng), **tổng toạ độ** (trung điểm); **tuyệt đối tránh `sqrt` và chia**.
- Đầu vào **không thứ tự** ⇒ dùng chữ ký bất biến theo thứ tự (khoảng cách đã sắp), hoặc sắp xếp rồi chuẩn hóa.
- Chỉ **4 điểm** ⇒ có thể thử vét cạn 24 hoán vị mà vẫn `O(1)`; thích hợp để làm oracle.

**Checklist điều kiện "hình vuông":**

| Thành phần | Cách kiểm tra bằng số nguyên |
|---|---|
| cạnh dương | `d_min > 0` |
| 4 cạnh bằng | `d[0] == d[3]` |
| 2 chéo bằng | `d[4] == d[5]` |
| góc vuông | `d[4] == 2 * d[0]` (Pythagoras) hoặc `dot == 0` |

**Mẹo xoay 90 độ:** `(x, y) → (-y, x)` để sinh hình vuông từ một cạnh — dùng làm test ngẫu nhiên.

**Họ bài liên quan:**

| Bài | Ý tưởng |
|---|---|
| 469. Convex Polygon | tích có hướng, đa giác lồi |
| 939. Minimum Area Rectangle | hình chữ nhật song song trục |
| 963. Minimum Area Rectangle II | chữ nhật bất kỳ: bình phương khoảng cách, trung điểm, đường chéo |
| 2013. Detect Squares | đếm hình vuông có sẵn điểm |
| 149. Max Points on a Line | thẳng hàng, hệ số góc nguyên |

## 15. Biến thể

### 15.1. Hình chữ nhật hợp lệ (4 điểm)

Chữ nhật = hình bình hành + hai đường chéo bằng nhau. Chỉ cần bỏ điều kiện vuông góc ở cách 4 (và cho phép dài khác nhau giữa hai cặp cạnh). Dùng đường chéo từ `p1` đến 3 ứng viên.

In [26]:
def is_rectangle(pts):
    p1, p2, p3, p4 = pts
    ring = [p1, p2, p3, p4]
    for k in (1, 2, 3):
        q = ring[k]
        r, s = [ring[i] for i in (1, 2, 3) if i != k]
        same_mid = p1[0] + q[0] == r[0] + s[0] and p1[1] + q[1] == r[1] + s[1]
        u, v = vec(p1, q), vec(r, s)
        if same_mid and dot(u, u) == dot(v, v) > 0:
            # hình bình hành + hai đường chéo bằng nhau, nhưng phải loại trường hợp 4 điểm thẳng hàng
            cross = u[0] * v[1] - u[1] * v[0]
            if cross != 0:
                return True
    return False

def is_rectangle_bruteforce(pts):
    for a, b, c, d in permutations(pts):
        e = [vec(a, b), vec(b, c), vec(c, d), vec(d, a)]
        if all(dot(v, v) > 0 for v in e) and all(dot(e[i], e[(i + 1) % 4]) == 0 for i in range(4)):
            return True
    return False

rng = random.Random(4)
cnt = 0
for quad in combinations_with_replacement([[x, y] for x in range(4) for y in range(4)], 4):
    q = list(quad)
    rng.shuffle(q)
    a, b = is_rectangle(q), is_rectangle_bruteforce(q)
    assert a == b, q
    cnt += a
print("chữ nhật (trên lưới 4x4):", cnt, "bộ; ví dụ:", is_rectangle([[0, 0], [3, 0], [3, 1], [0, 1]]), is_rectangle([[0, 0], [3, 0], [4, 1], [1, 1]]))

chữ nhật (trên lưới 4x4): 44 bộ; ví dụ: True False


### 15.2. Hình thoi hợp lệ

Hình thoi = hình bình hành + hai đường chéo **vuông góc** (độ dài hai đường chéo không cần bằng nhau). Cùng khung cách 4, đổi điều kiện "bằng nhau" thành "dương cả hai và vuông góc".

In [27]:
def is_rhombus(pts):
    p1, p2, p3, p4 = pts
    ring = [p1, p2, p3, p4]
    for k in (1, 2, 3):
        q = ring[k]
        r, s = [ring[i] for i in (1, 2, 3) if i != k]
        same_mid = p1[0] + q[0] == r[0] + s[0] and p1[1] + q[1] == r[1] + s[1]
        u, v = vec(p1, q), vec(r, s)
        if same_mid and dot(u, u) > 0 and dot(v, v) > 0 and dot(u, v) == 0:
            return True
    return False

def is_rhombus_bruteforce(pts):
    for a, b, c, d in permutations(pts):
        e = [vec(a, b), vec(b, c), vec(c, d), vec(d, a)]
        L = [dot(v, v) for v in e]
        # 4 cạnh bằng nhau và dương, và không suy biến (2 đường chéo không cùng phương)
        if L[0] > 0 and len(set(L)) == 1:
            u, v = vec(a, c), vec(b, d)
            if u[0] * v[1] - u[1] * v[0] != 0:
                return True
    return False

rng = random.Random(5)
cnt = 0
for quad in combinations_with_replacement([[x, y] for x in range(4) for y in range(4)], 4):
    q = list(quad)
    rng.shuffle(q)
    a, b = is_rhombus(q), is_rhombus_bruteforce(q)
    assert a == b, q
    cnt += a
print("hình thoi (trên lưới 4x4):", cnt, "bộ; hình thoi cạnh 5:", is_rhombus([[0, 0], [3, 4], [8, 4], [5, 0]]),
      "| chữ nhật 3x1:", is_rhombus([[0, 0], [3, 0], [3, 1], [0, 1]]))

hình thoi (trên lưới 4x4): 22 bộ; hình thoi cạnh 5: True | chữ nhật 3x1: False


### 15.3. Đếm số hình vuông trong một tập điểm (mọi hướng)

Với `n` điểm: brute force thử `C(n, 4)` bộ bốn. Cách hay hơn là **coi mỗi cặp là một đường chéo**: biết hai đầu `A, C` của đường chéo, hai đỉnh còn lại **được xác định duy nhất** bằng xoay quanh trung điểm. Dùng toạ độ gấp đôi để khỏi chia 2 (chỉ khi các số chẵn thì đỉnh có toạ độ nguyên). Mỗi hình vuông có 2 đường chéo nên đếm 2 lần.

In [28]:
def count_squares_bruteforce(points):
    return sum(Solution().validSquare(*q) for q in combinations(points, 4))

def count_squares_hash(points):
    s = set(map(tuple, points))
    pts = list(s)
    count = 0
    for a, c in combinations(pts, 2):
        sx, sy = a[0] + c[0], a[1] + c[1]               # tổng toạ độ = 2 * trung điểm
        dx, dy = c[0] - a[0], c[1] - a[1]               # vector đường chéo
        bx2, by2 = sx - dy, sy + dx                     # 2 * đỉnh b (xoay 90 độ quanh trung điểm)
        dx2, dy2 = sx + dy, sy - dx                     # 2 * đỉnh d
        if bx2 % 2 or by2 % 2:                          # toạ độ không nguyên: không có hình vuông
            continue
        if (bx2 // 2, by2 // 2) in s and (dx2 // 2, dy2 // 2) in s:
            count += 1
    return count // 2                                   # mỗi hình vuông có 2 đường chéo

rng = random.Random(6)
for _ in range(60):
    k = rng.randint(4, 12)
    pts = [[rng.randint(0, 5), rng.randint(0, 5)] for _ in range(k)]
    pts = [list(p) for p in set(map(tuple, pts))]            # bỏ điểm trùng
    assert count_squares_bruteforce(pts) == count_squares_hash(pts), pts
full = [[x, y] for x in range(4) for y in range(4)]
print("lưới 4x4 có", count_squares_hash(full), "hình vuông (mọi hướng); brute force:", count_squares_bruteforce(full))

lưới 4x4 có 20 hình vuông (mọi hướng); brute force: 20


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. 24 hoán vị | thử mọi thứ tự đỉnh, kiểm 4 cạnh bằng + góc vuông | `O(1)` (hằng lớn) | `O(1)` | Làm **oracle**; ít tin được nhưng chắc đúng |
| 2. 6 khoảng cách | chữ ký `d,d,d,d,2d,2d` sau khi sắp | `O(1)` | `O(1)` | **Nộp bài**: ngắn, bất biến thứ tự |
| 3. Bộ ba | mọi bộ ba là tam giác vuông cân | `O(1)` | `O(1)` | Khi nhớ định nghĩa "tam giác vuông cân", dễ chứng minh |
| 4. Đường chéo | cùng trung điểm + bằng nhau + vuông góc | `O(1)` | `O(1)` | Mở rộng sang chữ nhật / thoi / bình hành |
| 5. Sắp xếp một thứ tự | sắp điểm, chỉ thử `a→b→d→c` | `O(1)` | `O(1)` | Khi muốn kiểm tra theo định nghĩa gốc nhưng không thử 24 thứ tự |

Cả năm cách đều `O(1)` với 4 điểm; khác nhau về độ gọn, độ dễ sai và khả năng mở rộng.

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Các cell `TODO` chỉ **định nghĩa** hàm, không tự gọi, để notebook luôn chạy sạch.

### Bài 1 (đoán output)

Không chạy máy, dự đoán rồi đối chiếu đáp án ở mục 19:

```python
Solution().validSquare([0,0],[0,0],[0,0],[0,0])
Solution().validSquare([0,0],[2,0],[2,2],[0,2])
Solution().validSquare([0,0],[1,0],[2,0],[3,0])
Solution().validSquare([0,0],[2,1],[1,3],[-1,2])
```

### Bài 2 (tính tay)

Với 4 điểm `(0,0), (2,1), (1,3), (-1,2)` hãy tính tay 6 bình phương khoảng cách, sắp xếp, rồi nêu kết luận theo chữ ký `d,d,d,d,2d,2d`.

### Bài 3 (điền TODO)

Hoàn thành hai hàm `sq_dist_todo` (bình phương khoảng cách) và `is_square_todo` (dùng chữ ký 6 khoảng cách).

In [29]:
def sq_dist_todo(p, q):
    # TODO: trả về bình phương khoảng cách giữa p và q
    return None

def is_square_todo(p1, p2, p3, p4):
    # TODO: tính 6 khoảng cách bằng sq_dist_todo, sắp tăng dần, rồi kiểm tra 4 điều kiện
    return None
# Không gọi các hàm này ở đây. Hãy sửa rồi tự gọi ở một cell mới.

### Bài 4 (tự viết lại)

Không nhìn lại lời giải, hãy viết `is_square_triples(p1, p2, p3, p4)` theo **Cách 3**: mọi bộ ba đỉnh là tam giác vuông cân (`a > 0`, `a == b`, `a + b == c` sau khi sắp ba cạnh²).

In [30]:
def is_square_triples(p1, p2, p3, p4):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

### Bài 5 (mini challenge)

Viết `valid_rectangle(p1, p2, p3, p4)` trả về `True` nếu 4 điểm (không thứ tự) tạo thành **hình chữ nhật** (có thể không vuông, 4 góc vuông, cạnh dương). Kiểm chứng bằng vét cạn 24 hoán vị. (Chú ý: hãy nghĩ tới trường hợp 4 điểm **thẳng hàng** hoặc **trùng nhau**.)

In [31]:
def valid_rectangle(p1, p2, p3, p4):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: 4 điểm trùng nhau thì cạnh dài bao nhiêu?
- Mức 2: `(0,0),(2,0),(2,2),(0,2)` là hình vuông thẳng cạnh 2; còn `(0,0),(1,0),(2,0),(3,0)` thì 4 điểm cùng một đường thẳng.
- Mức 3: đáp án `False, True, False, True`; điểm thứ tư `(2,1)` có vector `(2,1)`, xoay 90 độ được `(-1,2)`.

**Bài 2**

- Mức 1: có `C(4,2) = 6` cặp.
- Mức 2: `(0,0)-(2,1)`: `4+1 = 5`; `(2,1)-(1,3)`: `1+4 = 5`; ...
- Mức 3: 4 cạnh đều `5`, 2 đường chéo đều `10 = 2 × 5` ⇒ sắp: `[5,5,5,5,10,10]` ⇒ hình vuông.

**Bài 3**

- Mức 1: `(p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2`.
- Mức 2: dùng `combinations` hoặc liệt kê tay 6 cặp rồi `sorted`.
- Mức 3: `d[0] > 0 and d[0] == d[3] and d[4] == d[5] and d[4] == 2 * d[0]`.

**Bài 4**

- Mức 1: một bộ ba có 3 cạnh; sắp tăng dần được `a ≤ b ≤ c`.
- Mức 2: tam giác vuông cân (tính theo bình phương cạnh) thỏa `a == b` và `a + b == c`, và `a > 0`.
- Mức 3: kiểm cả 4 bộ ba `(1,2,3), (1,2,4), (1,3,4), (2,3,4)`.

**Bài 5**

- Mức 1: chữ nhật là hình bình hành có hai đường chéo bằng nhau.
- Mức 2: thử 3 cách ghép `p1` với `q`; so tổng toạ độ cặp còn lại để biết cùng trung điểm.
- Mức 3: loại 4 điểm thẳng hàng bằng tích có hướng của hai vector đường chéo `u × v ≠ 0`; kiểm lại với vét cạn các hoán vị (4 cạnh dương + 4 góc vuông).

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert Solution().validSquare([0, 0], [0, 0], [0, 0], [0, 0]) is False
assert Solution().validSquare([0, 0], [2, 0], [2, 2], [0, 2]) is True
assert Solution().validSquare([0, 0], [1, 0], [2, 0], [3, 0]) is False
assert Solution().validSquare([0, 0], [2, 1], [1, 3], [-1, 2]) is True
print("Bài 1: False, True, False, True")

# Bài 2
pts = [[0, 0], [2, 1], [1, 3], [-1, 2]]
ds = sorted(sq_dist(a, b) for a, b in combinations(pts, 2))
print("Bài 2: các bình phương khoảng cách đã sắp:", ds)
assert ds == [5, 5, 5, 5, 10, 10]

Bài 1: False, True, False, True
Bài 2: các bình phương khoảng cách đã sắp: [5, 5, 5, 5, 10, 10]


In [33]:
# Bài 3
def sq_dist_answer(p, q):
    return (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2

def is_square_answer(p1, p2, p3, p4):
    d = sorted(sq_dist_answer(a, b) for a, b in combinations([p1, p2, p3, p4], 2))
    return d[0] > 0 and d[0] == d[3] and d[4] == d[5] and d[4] == 2 * d[0]

# Bài 4
def is_square_triples_answer(p1, p2, p3, p4):
    def ok(x, y, z):
        a, b, c = sorted([sq_dist_answer(x, y), sq_dist_answer(y, z), sq_dist_answer(x, z)])
        return a > 0 and a == b and a + b == c
    return ok(p1, p2, p3) and ok(p1, p2, p4) and ok(p1, p3, p4) and ok(p2, p3, p4)

# Bài 5
def valid_rectangle_answer(p1, p2, p3, p4):
    pts = [p1, p2, p3, p4]
    for k in (1, 2, 3):
        q = pts[k]
        r, s = [pts[i] for i in (1, 2, 3) if i != k]
        if p1[0] + q[0] != r[0] + s[0] or p1[1] + q[1] != r[1] + s[1]:
            continue
        u, v = vec(p1, q), vec(r, s)
        if dot(u, u) == dot(v, v) > 0 and u[0] * v[1] - u[1] * v[0] != 0:
            return True
    return False

rng = random.Random(1999)
for _ in range(2000):
    lim = rng.choice([1, 2, 3, 6])
    pts = [[rng.randint(-lim, lim), rng.randint(-lim, lim)] for _ in range(4)]
    expected = valid_square_bruteforce(*pts)
    assert is_square_answer(*pts) == expected
    assert is_square_triples_answer(*pts) == expected
    assert valid_rectangle_answer(*pts) == is_rectangle_bruteforce(pts)
print("Bài 3, 4, 5: OK")
print("ví dụ chữ nhật:", valid_rectangle_answer([0, 0], [3, 0], [3, 1], [0, 1]), "| thẳng hàng:", valid_rectangle_answer([0, 0], [1, 0], [2, 0], [3, 0]))

Bài 3, 4, 5: OK
ví dụ chữ nhật: True | thẳng hàng: False


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chữ ký 6 khoảng cách):**

```python
class Solution:
    def validSquare(self, p1: List[int], p2: List[int], p3: List[int], p4: List[int]) -> bool:
        def dist(a, b):
            return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2     # bình phương: số nguyên, không sqrt

        d = sorted([dist(p1, p2), dist(p1, p3), dist(p1, p4),
                    dist(p2, p3), dist(p2, p4), dist(p3, p4)])
        return d[0] > 0 and d[0] == d[3] and d[4] == d[5] and d[4] == 2 * d[0]
```

**Câu thần chú:** *"Sáu khoảng cách, sắp lên: bốn bằng nhau dương, hai chéo bằng nhau và gấp đôi."*

**Checklist trước khi submit:**

- [ ] Đề **không thứ tự** các điểm: không giả định `p1→p2→p3→p4` là vòng quanh.
- [ ] Dùng **bình phương khoảng cách** (số nguyên), không `sqrt`, không `==` trên số thực.
- [ ] Có điều kiện **cạnh dương** `d[0] > 0` (bốn điểm trùng nhau).
- [ ] Nếu dùng trung điểm: so sánh **tổng toạ độ**, không chia / không `//`.
- [ ] Đã thử: 4 điểm trùng, 3 điểm trùng, 4 điểm thẳng hàng, hình thoi, chữ nhật, hình vuông nghiêng, biên `±10^4`.
- [ ] Đã đối chiếu với brute force 24 hoán vị trên lưới nhỏ.
- [ ] Độ phức tạp: `O(1)` thời gian và bộ nhớ.